# DynaFeatureDock Pipeline

Tests whether Dyna-1 flexibility features improve FeatureDock docking on **high-flexibility pockets**.

**Before running:** activate the `featdock` conda environment in your terminal:
```
conda activate featdock
jupyter notebook
```

### Steps
1. Setup & imports
2. Configuration (set your paths here)
3. Compute Dyna-1 flexibility scores
4. Augment FEATURE vectors with dynamics token
5. Compute pocket-level flexibility & partition subsets
6. Train DynaFeatureDock (+ ablation without dynamics)
7. Evaluate on high-flex vs low-flex subsets
8. Score OpenADMET candidates
9. Results summary & plots

## 1. Setup & imports

In [3]:
!pip install rdkit

   ---------------------------------------- 0.0/24.6 MB ? eta -:--:--
   ---------- ----------------------------- 6.6/24.6 MB 36.0 MB/s eta 0:00:01
   --------------------- ------------------ 13.4/24.6 MB 33.5 MB/s eta 0:00:01
   ---------------------------------- ----- 21.0/24.6 MB 34.8 MB/s eta 0:00:01
   ---------------------------------------  24.4/24.6 MB 34.5 MB/s eta 0:00:01
   ---------------------------------------- 24.6/24.6 MB 29.3 MB/s eta 0:00:00


In [19]:
import os, sys, pickle, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
warnings.filterwarnings('ignore')

# Add project root and dyna module to path
PROJECT_ROOT = Path(os.getcwd()).parent  # featuredock-main/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
DYNA_DIR = PROJECT_ROOT / 'dyna_featuredock'
if str(DYNA_DIR) not in sys.path:
    sys.path.insert(0, str(DYNA_DIR))

# Check key dependencies
for pkg in ['torch', 'rdkit', 'sklearn', 'scipy', 'Bio']:
    try:
        __import__(pkg)
        print(f'  OK  {pkg}')
    except ImportError:
        print(f'  MISSING  {pkg}  -- run: conda activate featdock')

print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Project root: {PROJECT_ROOT}')

  OK  torch
  OK  rdkit
  OK  sklearn
  OK  scipy
  OK  Bio

PyTorch: 2.9.1+cpu
CUDA available: False
Project root: c:\Users\alira3\OneDrive - UW-Madison\featuredock-main\featuredock-main


## 1b. Dyna-1 setup (run once)

Before computing flexibility scores you need:
1. **Dyna-1 weights** (`dyna1.pt`) — free download from HuggingFace `gelnesr/Dyna-1`
2. **ESM3 access** — request at `huggingface.co/EvolutionaryScale/esm3-sm-open-v1`, then `huggingface-cli login`
3. **Extra packages** — installed in the cell below

In [21]:

# ── Install Dyna-1 dependencies into current env ────────────────────────────
# Only needs to run once; safe to re-run.
import subprocess, sys

DYNA1_REPO = PROJECT_ROOT / 'dyna_featuredock' / 'Dyna-1-main' / 'Dyna-1-main'

pkgs = [
    "huggingface_hub",
    "transformers<4.47.0",
    "einops",
    "biotite==0.41.2",
    "msgpack-numpy",
    "mdtraj",
    "MDAnalysis",
    "tenacity",
    "torcheval",
]
print("Installing Dyna-1 dependencies...")

# Rewrite: safer pip install with error handling and better feedback
try:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install"] + pkgs,
        check=True,
        capture_output=True,
        text=True,
    )
    print(result.stdout)
    print("Dependencies installed successfully.")
except subprocess.CalledProcessError as e:
    print("ERROR: pip install failed with exit code", e.returncode)
    print("Command:", " ".join(e.cmd))
    print("------ stdout ------\n", e.stdout)
    print("------ stderr ------\n", e.stderr)
    print(
        "\nSome dependencies failed to install. "
        "Check the error messages above.\n"
        "Common causes include package conflicts, conda env issues, or missing compilers.\n"
        "Try running the command below in your terminal for more verbose output and to debug interactively:\n"
        f"{sys.executable} -m pip install {' '.join(pkgs)}"
    )

# Add Dyna-1 repo to sys.path so its local esm/ and model/ modules are importable
if str(DYNA1_REPO) not in sys.path:
    sys.path.insert(0, str(DYNA1_REPO))
print("Done.")

# ── Check weights ────────────────────────────────────────────────────────────
WEIGHTS_PATH = DYNA1_REPO / 'model' / 'weights' / 'dyna1.pt'
if WEIGHTS_PATH.exists():
    print(f"✓  Dyna-1 weights found: {WEIGHTS_PATH}")
else:
    print(f"✗  Weights NOT found at {WEIGHTS_PATH}")
    print("   Run the next cell to download them.")


Installing Dyna-1 dependencies...
ERROR: pip install failed with exit code 1
Command: c:\Users\alira3\AppData\Local\anaconda3\python.exe -m pip install huggingface_hub transformers<4.47.0 einops biotite==0.41.2 msgpack-numpy mdtraj MDAnalysis tenacity torcheval
------ stdout ------
  Using cached huggingface_hub-1.20.0-py3-none-any.whl.metadata (14 kB)
  Using cached transformers-4.46.3-py3-none-any.whl.metadata (44 kB)
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
  Using cached biotite-0.41.2.tar.gz (17.4 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'

------ stderr ------
   error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ ex

In [22]:

# ── Download Dyna-1 weights (run once, ~300 MB) ──────────────────────────────
# Requires:
#   1. HuggingFace account with access to gelnesr/Dyna-1
#   2. ESM3 access approved at huggingface.co/EvolutionaryScale/esm3-sm-open-v1
#   3. huggingface-cli login  (paste your token when prompted)
#
# If you already have the weights file, copy dyna1.pt to:
#   dyna_featuredock/Dyna-1-main/Dyna-1-main/model/weights/dyna1.pt

if not WEIGHTS_PATH.exists():
    print("Downloading Dyna-1 weights from HuggingFace...")
    print("(You must be logged in: run  huggingface-cli login  in a terminal first)")
    from dyna1_predictor import download_weights
    # Pass your HF token as a string if not already logged in:
    # download_weights(hf_token="hf_YOUR_TOKEN_HERE")
    download_weights()
else:
    print(f"Weights already present at:\n  {WEIGHTS_PATH}")

# ── Quick smoke test with one PDB file ───────────────────────────────────────
# Uncomment to test on the example PDB bundled with FeatureDock:
# from dyna1_predictor import get_flexibility, _dyna1_root
# test_pdb = str(PROJECT_ROOT / 'examples' / '1b38.pdb')
# scores = get_flexibility('1b38', test_pdb, method='dyna1')
# print(f"1b38: {len(scores)} residues scored, mean={sum(scores.values())/len(scores):.3f}")


(You must be logged in: run  huggingface-cli login  in a terminal first)


ImportError: cannot import name 'download_weights' from 'dyna1_predictor' (c:\Users\alira3\OneDrive - UW-Madison\featuredock-main\featuredock-main\dyna_featuredock\dyna1_predictor.py)

## 2. Configuration
**Edit the paths in this cell to match your setup.**

In [ ]:

# ── PDBBind P-L folder (nested: P-L/P-L/{year-range}/{pdbid}/) ──────────────
PL_ROOT = str(PROJECT_ROOT / 'dyna_featuredock' / 'P-L' / 'P-L')

# Build a flat index {pdbid: path/to/pdbid_protein.pdb} once, then reuse
import glob as _glob

def build_pdb_index(pl_root):
    """Scan all year-range subfolders and return {pdbid: protein_pdb_path}."""
    idx = {}
    for prot in _glob.glob(os.path.join(pl_root, '*', '*', '*_protein.pdb')):
        pid = os.path.basename(prot).replace('_protein.pdb', '')
        idx[pid] = prot
    return idx

print('Building PDB file index (scans P-L folder once)...')
PDB_INDEX = build_pdb_index(PL_ROOT)
print(f'Found {len(PDB_INDEX)} protein PDB files in P-L/')

def get_pdb_path(pdbid):
    """Return the protein PDB path for a given PDB ID, or None if not found."""
    return PDB_INDEX.get(pdbid.lower())

# ── Refined-set PDB IDs ──────────────────────────────────────────────────────
REFINED_XLS = str(PROJECT_ROOT / 'data' / 'PDBBind_refined_set_20230524_032559.xlsx')
_ref_df = pd.read_excel(REFINED_XLS, header=1)
REFINED_IDS = set(_ref_df['PDB code'].dropna().str.strip().str.lower().tolist())
print(f'Refined set: {len(REFINED_IDS)} structures in xlsx')

# Only keep refined IDs that actually have a PDB file in P-L/
REFINED_IDS = {p for p in REFINED_IDS if get_pdb_path(p) is not None}
print(f'Refined set with PDB file present: {len(REFINED_IDS)}')

# ── Other paths ──────────────────────────────────────────────────────────────
PDBIDS_FILE    = str(PROJECT_ROOT / 'data' / 'labeled_pdblist.txt')
SPLIT_PKL      = str(PROJECT_ROOT / 'data' / 'train_val_test_split_refined.pkl')
CACHE_DIR      = str(DYNA_DIR / 'cache' / 'dyna1')
AUGMENTED_DIR  = str(DYNA_DIR / 'augmented_pvars')
MODEL_OUT_DIR  = str(PROJECT_ROOT / 'results' / 'dyna_model')

# ── Dyna-1 method ─────────────────────────────────────────────────────────────
# 'csv'     → load pre-computed CSVs from Make_Dyna1.ipynb (recommended)
# 'bfactor' → use B-factors as proxy (fast, no Dyna-1 install needed)
# 'dyna1'   → run local Dyna-1 ESM3 (requires HF token + dyna1.pt)
# 'auto'    → try dyna1, fall back to bfactor
DYNA_METHOD = 'csv'

# ── Path to pre-computed Dyna-1 CSVs from Make_Dyna1.ipynb ─────────────────
# Set this to the folder you downloaded from Google Colab
# Each file is named:  {pdbid}-Dyna1.csv
# Columns:  position, resnum, residue, p_exchange
DYNA1_CSV_DIR = str(DYNA_DIR / 'dyna1_csvs')   # <-- update if placed elsewhere

# ── Hyperparameters ──────────────────────────────────────────────────────────
USE_GPU       = torch.cuda.is_available()
DEVICE        = 'cuda' if USE_GPU else 'cpu'
HIDDEN_SIZE   = 20
N_LAYERS      = 5
N_HEADS       = 4
EPOCHS        = 50
LR            = 1e-4
BATCH_SIZE    = 32
N_RESAMPLES   = 2000
SEEDS         = [0, 42, 1234]
PATIENCE      = 10

HIGH_FLEX_QUANTILE = 0.75
POCKET_CUTOFF_ANG  = 10.0

for d in [CACHE_DIR, AUGMENTED_DIR, MODEL_OUT_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'\nDevice       : {DEVICE}')
print(f'Split        : {SPLIT_PKL}')
print(f'Dyna method  : {DYNA_METHOD}')
if DYNA_METHOD == 'csv':
    n_csvs = len(_glob.glob(os.path.join(DYNA1_CSV_DIR, '*-Dyna1.csv')))
    print(f'CSV dir      : {DYNA1_CSV_DIR}  ({n_csvs} files found)')


In [24]:

# ── Build ALL_PDBIDS: refined set only, PDB file present ────────────────────
with open(PDBIDS_FILE) as f:
    labeled = [l.strip().lower() for l in f if l.strip()]

# Intersect labeled list with refined set that has a PDB file
ALL_PDBIDS = [p for p in labeled if p in REFINED_IDS]
print(f'Labeled list       : {len(labeled)} total')
print(f'Refined + PDB file : {len(ALL_PDBIDS)} kept')

# ── Reproduce FeatureDock split: 10% of clusters → test (~660 structures) ───
if not os.path.exists(SPLIT_PKL):
    print('\nGenerating FeatureDock-style split from ClanGraph_90_df.pkl ...')
    clan_pkl = str(PROJECT_ROOT / 'data' / 'ClanGraph_90_df.pkl')
    with open(clan_pkl, 'rb') as f:
        clan_df = pickle.load(f)

    refined_set = set(ALL_PDBIDS)

    # Map each cluster to its refined-set members
    clan_to_ids = {}
    for _, row in clan_df.iterrows():
        cid  = row['Structure_Clan_ID']
        ids  = [p for p in row['PDBIDList'] if p in refined_set]
        if ids:
            clan_to_ids[cid] = ids

    total_clans = len(clan_to_ids)
    print(f'Clusters containing refined structures: {total_clans}')
    # FeatureDock paper: 1326 clusters, 10% test → ~133 test clusters → ~660 structures

    np.random.seed(42)
    clans = list(clan_to_ids.keys())
    np.random.shuffle(clans)

    n_test  = max(1, round(total_clans * 0.10))   # 10% test
    n_val   = max(1, round(total_clans * 0.10))   # 10% val
    n_train = total_clans - n_test - n_val         # 80% train

    test_clans  = clans[:n_test]
    val_clans   = clans[n_test:n_test + n_val]
    train_clans = clans[n_test + n_val:]

    train_ids = [pid for c in train_clans for pid in clan_to_ids[c]]
    val_ids   = [pid for c in val_clans   for pid in clan_to_ids[c]]
    test_ids  = [pid for c in test_clans  for pid in clan_to_ids[c]]

    split = {'train': train_ids, 'val': val_ids, 'test': test_ids,
             'n_clusters': total_clans, 'n_test_clusters': n_test}
    with open(SPLIT_PKL, 'wb') as f:
        pickle.dump(split, f)
    print(f'Split saved → {SPLIT_PKL}')

with open(SPLIT_PKL, 'rb') as f:
    SPLITS = pickle.load(f)
TRAIN_IDS = SPLITS['train']
VAL_IDS   = SPLITS['val']
TEST_IDS  = SPLITS['test']

print(f'\nClusters total / test : {SPLITS.get("n_clusters","?")} / {SPLITS.get("n_test_clusters","?")}')
print(f'Train / Val / Test    : {len(TRAIN_IDS)} / {len(VAL_IDS)} / {len(TEST_IDS)}')
print(f'Test set target       : ~660  (FeatureDock paper)')


Labeled list       : 4516 total
Refined + PDB file : 4507 kept

Clusters total / test : 1323 / 132
Train / Val / Test    : 3815 / 334 / 357
Test set target       : ~660  (FeatureDock paper)


In [25]:
# Make a folder called "PDB_labled" and copy in all labeled PDB files
import shutil

PDB_LABELED_DIR = PROJECT_ROOT / "PDB_labled"
os.makedirs(PDB_LABELED_DIR, exist_ok=True)

for pid in ALL_PDBIDS:
    pdb_path = get_pdb_path(pid)
    if pdb_path is not None and os.path.exists(pdb_path):
        dest_path = PDB_LABELED_DIR / f"{pid}.pdb"
        shutil.copy2(pdb_path, dest_path)
    else:
        print(f"Warning: PDB file for {pid} not found, skipping.")
print(f"Copied {len(os.listdir(PDB_LABELED_DIR))} PDBs to {PDB_LABELED_DIR}")

Copied 4507 PDBs to c:\Users\alira3\OneDrive - UW-Madison\featuredock-main\featuredock-main\PDB_labled


In [29]:
import os
import pandas as pd

pdb_folder = '../PDB_labled'
csv_out = "PDBbinder.csv"

# Map 3-letter AA code to 1-letter code
aas = """A ALA 
C CYS
D ASP
E GLU
F PHE
G GLY
H HIS
I ILE
K LYS
L LEU
M MET
N ASN
P PRO
Q GLN
R ARG
S SER
T THR
V VAL
W TRP
Y TYR""".split('\n')
aa_dict = {}
for aa in aas:
    parts = aa.split()
    if len(parts) == 2:
        aa_dict[parts[1]] = parts[0]

records = []

for fname in os.listdir(pdb_folder):
    if fname.endswith('.pdb'):
        pdb_path = os.path.join(pdb_folder, fname)
        seq = ""
        name = os.path.splitext(fname)[0]  # use filename (without extension) as name
        with open(pdb_path) as file:
            # Get lines with alpha carbon (CA)
            lines = [line for line in file if ' CA ' in line]
            # Optionally, filter for chain A if desired:
            # lines = [line for line in lines if ' A ' in line]
            for line in lines:
                resname = line.split()[3]
                aa = aa_dict.get(resname, 'X')
                seq += aa
        records.append([name, seq])

df = pd.DataFrame(records, columns=["Uniprot_IDs", "Sequence"])
df.to_csv(csv_out, index=False)

In [28]:
ls

 Volume in drive C has no label.
 Volume Serial Number is 70EF-E5FD

 Directory of c:\Users\alira3\OneDrive - UW-Madison\featuredock-main\featuredock-main\dyna_featuredock

06/18/2026  11:47 AM    <DIR>          .
06/18/2026  01:22 PM    <DIR>          ..
06/08/2026  03:38 PM                78 __init__.py
06/16/2026  04:35 PM    <DIR>          __pycache__
06/08/2026  03:35 PM             6,975 augment_features.py
06/16/2026  04:09 PM    <DIR>          augmented_pvars
06/16/2026  04:35 PM    <DIR>          cache
06/08/2026  04:43 PM    <DIR>          dyna_featuredock
06/08/2026  03:35 PM             4,013 dyna_loaders.py
06/08/2026  03:35 PM             6,954 dyna_transformer.py
06/18/2026  11:47 AM             9,890 dyna1_predictor.py
06/18/2026  11:42 AM    <DIR>          Dyna-1-main
06/18/2026  11:42 AM        27,811,401 Dyna-1-main.zip
06/18/2026  02:07 PM            70,185 DynaFeatureDock_Pipeline.ipynb
06/08/2026  03:37 PM             5,272 eval_flexibility.py
06/18/2026  11:00 AM

## 3. Compute Dyna-1 flexibility scores

Produces `cache/dyna1/{pdbid}.dyna1.pkl` — `{(chain, resnum): flexibility_prob}` per structure.

- With `dyna1` package installed → real µs–ms motion probabilities  
- Otherwise → normalized backbone B-factors (fast proxy, runs on any machine)

In [ ]:

# ── Load Dyna-1 flexibility scores ───────────────────────────────────────────
# Supports three modes (set DYNA_METHOD in Cell-04):
#   'csv'     → read pre-computed CSVs from Make_Dyna1.ipynb (fastest, recommended)
#   'bfactor' → normalize backbone B-factors (no ML, works offline)
#   'dyna1' / 'auto' → run local Dyna-1 ESM3 subprocess

FLEX_ALL_PKL = str(DYNA_DIR / 'cache' / 'flexibility_all.pkl')


def load_flex_from_csvs(pdbids, csv_dir):
    """
    Read pre-computed Dyna-1 CSVs (from Make_Dyna1.ipynb).
    CSV columns: position, resnum, residue, p_exchange
    Returns {pdbid: {(chain, resnum): p_exchange}}.
    """
    flex = {}
    missing = []
    for pid in pdbids:
        csv_path = os.path.join(csv_dir, f'{pid}-Dyna1.csv')
        if not os.path.isfile(csv_path):
            missing.append(pid)
            continue
        try:
            df = pd.read_csv(csv_path)
            # Support both 'resnum' column (Make_Dyna1 output) and positional fallback
            if 'resnum' in df.columns:
                scores = {('A', int(row['resnum'])): float(row['p_exchange'])
                          for _, row in df.iterrows()}
            else:
                # Older format: use position as resnum
                scores = {('A', int(row['position'])): float(row['p_exchange'])
                          for _, row in df.iterrows()}
            flex[pid] = scores
        except Exception as e:
            print(f'  [{pid}] CSV read error: {e}')

    print(f'Loaded CSVs: {len(flex)} found, {len(missing)} missing')
    if missing[:5]:
        print(f'  Missing examples: {missing[:5]}')
    return flex


# ── Main: load or compute ─────────────────────────────────────────────────────
if os.path.exists(FLEX_ALL_PKL):
    with open(FLEX_ALL_PKL, 'rb') as f:
        flex_all = pickle.load(f)
    print(f'Loaded cached flexibility scores for {len(flex_all)} structures.')

elif DYNA_METHOD == 'csv':
    if not os.path.isdir(DYNA1_CSV_DIR):
        raise FileNotFoundError(
            f'DYNA1_CSV_DIR not found: {DYNA1_CSV_DIR}\n'
            'Run Make_Dyna1.ipynb on Colab first, then download the dyna1_csvs/ folder here.'
        )
    print(f'Loading pre-computed Dyna-1 CSVs from {DYNA1_CSV_DIR} ...')
    flex_all = load_flex_from_csvs(ALL_PDBIDS, DYNA1_CSV_DIR)

    # Save cache so subsequent runs are instant
    os.makedirs(os.path.dirname(FLEX_ALL_PKL), exist_ok=True)
    with open(FLEX_ALL_PKL, 'wb') as f:
        pickle.dump(flex_all, f)
    print(f'Cache saved → {FLEX_ALL_PKL}')

else:
    from dyna1_predictor import get_flexibility
    print(f'Computing flexibility for {len(ALL_PDBIDS)} structures (method={DYNA_METHOD})...')
    os.makedirs(CACHE_DIR, exist_ok=True)
    flex_all = {}
    for i, pid in enumerate(ALL_PDBIDS):
        pdbfile = get_pdb_path(pid)
        if pdbfile is None:
            continue
        try:
            scores = get_flexibility(pid, pdbfile, cache_dir=CACHE_DIR, method=DYNA_METHOD)
            flex_all[pid] = scores
        except Exception as e:
            print(f'  [{pid}] error: {e}')
        if (i + 1) % 200 == 0:
            print(f'  {i+1}/{len(ALL_PDBIDS)} done')
    with open(FLEX_ALL_PKL, 'wb') as f:
        pickle.dump(flex_all, f)
    print(f'Done. Saved to {FLEX_ALL_PKL}')

# ── Summary ───────────────────────────────────────────────────────────────────
if flex_all:
    sample_pid  = list(flex_all.keys())[0]
    sample_vals = list(flex_all[sample_pid].values())
    print(f'\nSample ({sample_pid}): {len(sample_vals)} residues, mean flex = {np.mean(sample_vals):.3f}')
    all_vals = [v for scores in flex_all.values() for v in scores.values()]
    print(f'Global: {len(flex_all)} structures, {len(all_vals):,} residues, mean = {np.mean(all_vals):.3f}')
else:
    print('WARNING: flex_all is empty. Check DYNA1_CSV_DIR or Dyna-1 setup.')


In [ ]:
# Plot per-residue flexibility distribution across all structures
all_vals = [v for scores in flex_all.values() for v in scores.values()]
plt.figure(figsize=(7, 3))
plt.hist(all_vals, bins=50, color='steelblue', edgecolor='white')
plt.xlabel('Flexibility score (0 = rigid, 1 = flexible)')
plt.ylabel('Count')
plt.title(f'Per-residue flexibility distribution (n={len(all_vals):,} residues)')
plt.tight_layout()
plt.show()

## 4. Augment FEATURE vectors with dynamics token

For each grid point, compute the mean Dyna-1 score of Cα atoms in each of the 6 shells.  
Appends a `(N, 6)` dynamics column → `(N, 486)` augmented feature file.

In [ ]:

from augment_features import augment_pvar

os.makedirs(AUGMENTED_DIR, exist_ok=True)
skipped, done = 0, 0
for i, pid in enumerate(ALL_PDBIDS):
    out_path  = os.path.join(AUGMENTED_DIR, f'{pid}.dyna.pvar')
    pvarfile  = os.path.join(AUGMENTED_DIR, f'{pid}.property.pvar')   # pre-computed FEATURE vec
    voxelfile = os.path.join(AUGMENTED_DIR, f'{pid}.voxels.pkl')
    pdbfile   = get_pdb_path(pid)

    if os.path.exists(out_path):
        done += 1; continue
    if not (os.path.exists(pvarfile) and os.path.exists(voxelfile)) or pdbfile is None:
        skipped += 1; continue

    flex = flex_all.get(pid, {})
    try:
        augment_pvar(pvarfile, voxelfile, pdbfile, flex, out_pvarfile=out_path)
        done += 1
    except Exception as e:
        print(f'  [{pid}] {e}'); skipped += 1
    if (i + 1) % 200 == 0:
        print(f'  {i+1}/{len(ALL_PDBIDS)}  done={done}  skipped={skipped}')

print(f'\nAugmentation complete: {done} augmented, {skipped} skipped (missing .pvar/.voxels)')
print('NOTE: .pvar files are generated by FeatureDock step2 (featurize.py).')
print('If skipped > 0, run the FeatureDock featurization pipeline first.')

# Verify one file if any exist
sample_dyna = os.path.join(AUGMENTED_DIR, f'{ALL_PDBIDS[0]}.dyna.pvar')
if os.path.exists(sample_dyna):
    with open(sample_dyna, 'rb') as f: arr = pickle.load(f)
    print(f'Sample shape: {arr.shape}  (expected N×486)')


## 5. Pocket flexibility scores & subset partitioning

Each structure gets a scalar = mean flexibility of residues within `POCKET_CUTOFF_ANG` Å of the ligand centroid.  
Top 25% → **high-flex** subset; bottom 75% → **low-flex** subset.

In [ ]:

from flexibility_subset import pocket_flexibility, partition_by_flexibility

POCKET_SCORES_PKL = str(DYNA_DIR / 'pocket_flexibility.pkl')

if os.path.exists(POCKET_SCORES_PKL):
    with open(POCKET_SCORES_PKL, 'rb') as f:
        pocket_data = pickle.load(f)
    pocket_scores = pocket_data['scores']
    print(f'Loaded cached pocket scores for {len(pocket_scores)} structures.')
else:
    print('Computing pocket-level flexibility scores...')
    pocket_scores = {}
    for pid in ALL_PDBIDS:
        pdbfile = get_pdb_path(pid)
        if pdbfile is None:
            continue
        flex = flex_all.get(pid, {})
        pocket_scores[pid] = pocket_flexibility(pid, None, flex,
                                                pocket_cutoff=POCKET_CUTOFF_ANG,
                                                _pdbfile_override=pdbfile)
    high_ids, low_ids = partition_by_flexibility(pocket_scores, HIGH_FLEX_QUANTILE)
    with open(POCKET_SCORES_PKL, 'wb') as f:
        pickle.dump({'scores': pocket_scores, 'high': high_ids, 'low': low_ids}, f)
    print(f'Saved to {POCKET_SCORES_PKL}')

high_ids, low_ids = partition_by_flexibility(pocket_scores, HIGH_FLEX_QUANTILE)

test_set  = set(TEST_IDS)
test_high = [p for p in high_ids if p in test_set]
test_low  = [p for p in low_ids  if p in test_set]
print(f'Test: {len(TEST_IDS)} total | {len(test_high)} high-flex | {len(test_low)} low-flex')


In [ ]:
# Plot pocket flexibility distribution
scores_arr = np.array([pocket_scores.get(p, 0.5) for p in ALL_PDBIDS])
cutoff = np.quantile(scores_arr, HIGH_FLEX_QUANTILE)

plt.figure(figsize=(7, 3))
plt.hist(scores_arr, bins=50, color='steelblue', edgecolor='white')
plt.axvline(cutoff, color='tomato', linewidth=2, label=f'Q{HIGH_FLEX_QUANTILE:.0%} = {cutoff:.3f}')
plt.xlabel('Mean pocket flexibility')
plt.ylabel('Structures')
plt.title('Pocket flexibility distribution (high-flex = right of red line)')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Train DynaFeatureDock

Trains two models:
- **DynaFeatureDock** — 81 tokens (80 physicochemical + 1 dynamics)
- **Ablation** — 80 tokens only (identical to original FeatureDock architecture)

Each is trained with 3 random seeds for stability.

In [ ]:
import random
from torch.utils.data import DataLoader
from torch.optim import Adam
import torch.nn as nn

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from models.earlystop import EarlyStopping
from dyna_loaders import DynaClassifierDataset
from dyna_transformer import DynaBertSentClassifier

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def collate_fn(batch):
    xs, ys = zip(*batch)
    return torch.cat(xs), torch.cat(ys)

def run_epoch(model, loader, criterion, optimizer, device, train=True):
    model.train(train)
    total_loss, n = 0.0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device).long()
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(y); n += len(y)
    return total_loss / n

def train_model(use_dynamics, seed, label):
    set_seed(seed)
    out_dir = os.path.join(MODEL_OUT_DIR, label)
    os.makedirs(out_dir, exist_ok=True)
    ckpt_path = os.path.join(out_dir, 'best_checkpoint.pt')
    config_path = os.path.join(out_dir, 'config.torch')

    ds_kw = dict(datadir=AUGMENTED_DIR, orig_datadir=VOXEL_DIR,
                 resample=True, n_resamples=N_RESAMPLES, use_dynamics=use_dynamics)
    train_ds = DynaClassifierDataset(pids=TRAIN_IDS, **ds_kw)
    val_ds   = DynaClassifierDataset(pids=VAL_IDS, resample=False,
                                     datadir=AUGMENTED_DIR, orig_datadir=VOXEL_DIR,
                                     use_dynamics=use_dynamics)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_fn, num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              collate_fn=collate_fn, num_workers=0)

    model = DynaBertSentClassifier(
        n_class=2, num_shells=6, feature_per_shell=80,
        hidden_size=HIDDEN_SIZE, intermediate_size=HIDDEN_SIZE*4,
        num_hidden_layers=N_LAYERS, num_attention_heads=N_HEADS,
        use_dynamics=use_dynamics,
    ).to(DEVICE).float()

    optimizer = Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    stopper   = EarlyStopping(patience=PATIENCE, verbose=False, path=ckpt_path)

    history = []
    for epoch in range(1, EPOCHS + 1):
        tl = run_epoch(model, train_loader, criterion, optimizer, DEVICE)
        vl = run_epoch(model, val_loader,   criterion, optimizer, DEVICE, train=False)
        history.append({'epoch': epoch, 'train_loss': tl, 'val_loss': vl})
        if epoch % 5 == 0:
            print(f'  [{label}] epoch {epoch:3d}  train={tl:.4f}  val={vl:.4f}')
        stopper(vl, model)
        if stopper.early_stop:
            print(f'  [{label}] early stop at epoch {epoch}')
            break

    torch.save({'model': model, 'args': dict(
        use_dynamics=use_dynamics, seed=seed, hidden_size=HIDDEN_SIZE
    )}, config_path)
    return history

print('Training functions ready.')

In [ ]:
# Train with dynamics (3 seeds)
dyna_histories = {}
for seed in SEEDS:
    label = f'dyna_seed{seed}'
    print(f'Training DynaFeatureDock seed={seed}...')
    dyna_histories[seed] = train_model(use_dynamics=True, seed=seed, label=label)
print('\nDynaFeatureDock training complete.')

In [ ]:
# Train ablation WITHOUT dynamics (1 seed)
print('Training ablation (no dynamics)...')
ablation_history = train_model(use_dynamics=False, seed=42, label='ablation_no_dyna')
print('Ablation training complete.')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for seed, hist in dyna_histories.items():
    df = pd.DataFrame(hist)
    axes[0].plot(df['epoch'], df['val_loss'], label=f'dyna seed={seed}')

ab_df = pd.DataFrame(ablation_history)
axes[0].plot(ab_df['epoch'], ab_df['val_loss'], '--', color='gray', label='ablation (no dyna)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val loss'); axes[0].set_title('Validation loss')
axes[0].legend(fontsize=8)

for seed, hist in dyna_histories.items():
    df = pd.DataFrame(hist)
    axes[1].plot(df['epoch'], df['train_loss'], label=f'dyna seed={seed}')
axes[1].plot(ab_df['epoch'], ab_df['train_loss'], '--', color='gray', label='ablation')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Train loss'); axes[1].set_title('Training loss')
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()

## 7. Evaluate on high-flex vs low-flex subsets

Reports **AUROC** and **KL divergence** (matching FeatureDock paper metrics) on:
- All test structures
- High-flexibility subset (top 25% pocket flex)
- Low-flexibility subset (bottom 75%)

In [ ]:
from sklearn.metrics import roc_auc_score
from scipy.stats import entropy

def predict_probs(model, pids, use_dynamics):
    ds = DynaClassifierDataset(
        datadir=AUGMENTED_DIR, orig_datadir=VOXEL_DIR,
        pids=pids, resample=False, use_dynamics=use_dynamics
    )
    loader = DataLoader(ds, batch_size=256, shuffle=False,
                        collate_fn=collate_fn, num_workers=0)
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            p = torch.softmax(model(x.to(DEVICE)), dim=1)[:, 1].cpu().numpy()
            all_probs.extend(p); all_labels.extend(y.numpy())
    return np.array(all_probs), np.array(all_labels)

def eval_subsets(model, use_dynamics):
    rows = []
    for label, pids in [('all', TEST_IDS), ('high_flex', test_high), ('low_flex', test_low)]:
        if len(pids) == 0: continue
        probs, labels = predict_probs(model, pids, use_dynamics)
        if len(np.unique(labels)) < 2:
            rows.append({'subset': label, 'n': len(pids), 'auc': float('nan'), 'kl': float('nan')})
            continue
        auc = roc_auc_score(labels, probs)
        p = np.clip(probs / probs.sum(), 1e-7, None)
        q = np.clip(labels / max(labels.sum(), 1e-7), 1e-7, None)
        kl = float(entropy(q, p))
        rows.append({'subset': label, 'n': len(pids), 'auc': auc, 'kl': kl})
    return pd.DataFrame(rows)

print('Evaluation functions ready.')

In [ ]:
results = {}

# Evaluate each dyna model seed
for seed in SEEDS:
    label = f'dyna_seed{seed}'
    cfg = torch.load(os.path.join(MODEL_OUT_DIR, label, 'config.torch'), map_location=DEVICE)
    model = cfg['model'].to(DEVICE).float()
    ckpt  = torch.load(os.path.join(MODEL_OUT_DIR, label, 'best_checkpoint.pt'), map_location=DEVICE)
    model.load_state_dict(ckpt)
    results[label] = eval_subsets(model, use_dynamics=True)

# Evaluate ablation
cfg_ab = torch.load(os.path.join(MODEL_OUT_DIR, 'ablation_no_dyna', 'config.torch'), map_location=DEVICE)
model_ab = cfg_ab['model'].to(DEVICE).float()
ckpt_ab  = torch.load(os.path.join(MODEL_OUT_DIR, 'ablation_no_dyna', 'best_checkpoint.pt'), map_location=DEVICE)
model_ab.load_state_dict(ckpt_ab)
results['ablation_no_dyna'] = eval_subsets(model_ab, use_dynamics=False)

# Average across dyna seeds
dyna_dfs = [results[f'dyna_seed{s}'] for s in SEEDS]
combined = pd.concat(dyna_dfs)
dyna_mean = combined.groupby('subset')[['auc', 'kl']].mean().reset_index()
dyna_mean['model'] = 'DynaFeatureDock (mean±std)'
dyna_std  = combined.groupby('subset')[['auc', 'kl']].std().reset_index()

ab_df = results['ablation_no_dyna'].copy()
ab_df['model'] = 'Ablation (no dynamics)'

print('=== DynaFeatureDock (mean across seeds) ===')
print(dyna_mean.to_string(index=False))
print('\n=== Ablation (no dynamics) ===')
print(ab_df[['subset','n','auc','kl']].to_string(index=False))

In [ ]:
# Bar plot: AUC comparison
subsets = ['all', 'high_flex', 'low_flex']
labels  = ['All', 'High-flex\n(top 25%)', 'Low-flex\n(bottom 75%)']

dyna_aucs = [dyna_mean[dyna_mean.subset==s]['auc'].values[0] for s in subsets]
dyna_stds = [dyna_std[dyna_std.subset==s]['auc'].values[0]   for s in subsets]
ab_aucs   = [ab_df[ab_df.subset==s]['auc'].values[0]         for s in subsets]

x = np.arange(len(subsets))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
bars1 = ax.bar(x - w/2, dyna_aucs, w, yerr=dyna_stds, capsize=4,
               color='steelblue', label='DynaFeatureDock')
bars2 = ax.bar(x + w/2, ab_aucs,   w,
               color='lightcoral', label='Ablation (no dynamics)')
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='Random')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('AUROC'); ax.set_ylim(0, 1)
ax.set_title('FeatureDock vs DynaFeatureDock: AUROC by pocket flexibility')
ax.legend()
plt.tight_layout(); plt.show()

# Interpretation
delta_high = dyna_aucs[1] - ab_aucs[1]
delta_low  = dyna_aucs[2] - ab_aucs[2]
print(f'\nAUROC gain from dynamics:')
print(f'  High-flex subset: {delta_high:+.4f}')
print(f'  Low-flex subset:  {delta_low:+.4f}')
if delta_high > delta_low + 0.01:
    print('  → Dynamics help more on flexible pockets (hypothesis supported)')
elif abs(delta_high - delta_low) < 0.01:
    print('  → No differential benefit by flexibility (dynamics may be redundant)')
else:
    print('  → Dynamics help more on rigid pockets (unexpected — check Dyna-1 quality)')

## 8. Score OpenADMET candidates

Uses the best DynaFeatureDock model to score ExpansionRx candidates with the envelope scoring function (FeatureDock Eq. 2/3).  
The score can be added as a feature column to the shin-chan ADMET ensemble.

In [ ]:
# Load OpenADMET dataset
OPENADMET_CSV = None   # set to local path if downloaded, else loads from HuggingFace

try:
    if OPENADMET_CSV and os.path.exists(OPENADMET_CSV):
        df_admet = pd.read_csv(OPENADMET_CSV)
    else:
        print('Downloading from HuggingFace...')
        df_admet = pd.read_csv(
            'hf://datasets/openadmet/openadmet-expansionrx-challenge-train-data/expansion_data_train.csv'
        )
    print(f'Loaded {len(df_admet)} compounds')
    print(df_admet.head(3))
except Exception as e:
    print(f'Could not load OpenADMET data: {e}')
    print('Install with: pip install huggingface_hub datasets')
    df_admet = None

In [ ]:
from openadmet_integration import smiles_to_heavy_atom_coords, score_ligand_against_envelope

# Pick best dyna model (seed 42)
best_model_dir = os.path.join(MODEL_OUT_DIR, 'dyna_seed42')
cfg_best = torch.load(os.path.join(best_model_dir, 'config.torch'), map_location=DEVICE)
model_best = cfg_best['model'].to(DEVICE).float()
ckpt_best  = torch.load(os.path.join(best_model_dir, 'best_checkpoint.pt'), map_location=DEVICE)
model_best.load_state_dict(ckpt_best)
model_best.eval()

def get_envelope(pdbid):
    """Load pre-computed voxel probs for a target pocket."""
    pvar = os.path.join(AUGMENTED_DIR, f'{pdbid}.dyna.pvar')
    vox  = os.path.join(VOXEL_DIR,     f'{pdbid}.voxels.pkl')
    if not (os.path.exists(pvar) and os.path.exists(vox)):
        return None, None
    with open(pvar, 'rb') as f: prop = pickle.load(f)
    with open(vox,  'rb') as f: coords = pickle.load(f)
    model_best.eval()
    probs = []
    with torch.no_grad():
        for start in range(0, len(prop), 4096):
            batch = torch.from_numpy(prop[start:start+4096].astype(np.float32)).to(DEVICE)
            p = torch.softmax(model_best(batch), dim=1)[:, 1].cpu().numpy()
            probs.extend(p)
    return coords, np.array(probs)

if df_admet is not None:
    # Detect SMILES and target columns
    smiles_col = next((c for c in df_admet.columns if 'smiles' in c.lower()), df_admet.columns[0])
    target_col = next((c for c in df_admet.columns if 'target' in c.lower()), None)
    print(f'SMILES column: {smiles_col}')
    print(f'Target column: {target_col}')
    print(df_admet[[smiles_col] + ([target_col] if target_col else [])].head())

In [ ]:
# Score a sample of OpenADMET compounds against available pocket envelopes
OPENADMET_OUT = str(DYNA_DIR / 'openadmet_dyna_scores.csv')

if df_admet is not None:
    envelope_cache = {}
    scores_out = []

    sample_df = df_admet.head(200)  # score first 200 rows; remove .head() for full dataset
    for _, row in sample_df.iterrows():
        smiles = row[smiles_col]
        target = row[target_col] if target_col else 'unknown'

        if target not in envelope_cache:
            envelope_cache[target] = get_envelope(str(target))

        v_coords, v_probs = envelope_cache[target]
        if v_coords is None:
            scores_out.append(float('nan'))
            continue

        lig_coords = smiles_to_heavy_atom_coords(smiles)
        if lig_coords is None:
            scores_out.append(float('nan'))
        else:
            scores_out.append(score_ligand_against_envelope(lig_coords, v_coords, v_probs))

    sample_df = sample_df.copy()
    sample_df['dyna_envelope_score'] = scores_out
    sample_df.to_csv(OPENADMET_OUT, index=False)

    n_scored = sum(1 for s in scores_out if not np.isnan(s))
    print(f'Scored {n_scored}/{len(sample_df)} compounds')
    print(f'Saved to {OPENADMET_OUT}')
    print(sample_df[['dyna_envelope_score']].describe())

## 9. Results summary

In [ ]:
# Full results table
rows = []
for seed in SEEDS:
    df = results[f'dyna_seed{seed}']
    for _, r in df.iterrows():
        rows.append({'model': f'DynaFeatureDock seed={seed}',
                     'subset': r['subset'], 'n': r['n'],
                     'auc': round(r['auc'], 4), 'kl': round(r['kl'], 4)})
for _, r in results['ablation_no_dyna'].iterrows():
    rows.append({'model': 'Ablation (no dyna)',
                 'subset': r['subset'], 'n': r['n'],
                 'auc': round(r['auc'], 4), 'kl': round(r['kl'], 4)})

summary = pd.DataFrame(rows)
summary_csv = str(DYNA_DIR / 'eval_results.csv')
summary.to_csv(summary_csv, index=False)
print(f'Saved to {summary_csv}')
summary

In [ ]:
print('=' * 55)
print('  DynaFeatureDock — Experiment Summary')
print('=' * 55)
print(f'  Structures processed   : {len(flex_all)}')
print(f'  Test set (total)       : {len(TEST_IDS)}')
print(f'  Test set (high-flex)   : {len(test_high)}')
print(f'  Test set (low-flex)    : {len(test_low)}')
print(f'  Flexibility cutoff     : {cutoff:.3f}')
print(f'  Dyna method            : {DYNA_METHOD}')
print()
print('  AUROC (mean across seeds):')
for s in subsets:
    row_dyna = dyna_mean[dyna_mean.subset==s]
    row_ab   = ab_df[ab_df.subset==s]
    if len(row_dyna) and len(row_ab):
        print(f'    {s:<12}  dyna={row_dyna.auc.values[0]:.4f}  ablation={row_ab.auc.values[0]:.4f}')
print('=' * 55)